# Mixing the design principle into Top-k and Tuned L1

Tests whether the identity-tracked, boundary-escalating principle behind Rate-KL is better framed as a standalone objective or as a PLUGGABLE AUXILIARY REGULARIZER: adds the log-barrier + budget penalty (already confirmed as a second working instantiation, `rate_barrier_sae.py`) on top of Top-k's and Tuned L1's existing objectives, unchanged otherwise.

**Predictions to check:**
- **Top-k + barrier**: real hypothesis. Top-k has no persistent per-feature identity constraint (hard selection is independent each step) and we already measured it isn't hub-immune (importance-Gini 0.58-0.72 across datasets) -- adding the barrier should raise purity without touching Top-k's sparsity mechanism.
- **Tuned L1 + barrier**: weaker hypothesis / control. L1's penalty is invariant to how mass is distributed (only the sum matters), so it never had positive pressure toward concentration -- if the barrier does little here, that's informative too (confirms the pathology is about objectives that reward concentration, not sparsity objectives generally). A 1-epoch smoke test already showed a larger-than-predicted purity jump for L1 too, worth checking whether that holds with real training.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# Small lambda_barrier sweep at fixed lambda_budget=0.5 (the value that worked
# well in the standalone barrier experiment), Fashion-MNIST, one seed.
for lb in [0.001, 0.005, 0.01]:
    !python mixed_regularizer_experiment.py --dataset fashion_mnist --seed 0 --rho 0.09 --lambda-barrier {lb} --lambda-budget 0.5

In [ ]:
!zip -r mixed_regularizer_results.zip results/mixed_regularizer
from google.colab import files
files.download('mixed_regularizer_results.zip')